# RECRAG 생성 모듈 — QLoRA 파인튜닝 + 정량 평가 (Colab T4, 원클릭)

위에서부터 셀을 **순서대로 실행**하면 다음이 모두 수행됩니다.

0. GPU 확인 → 1. repo clone + 설치 → 2~3. (선택) **누수 없는 데이터셋 재생성** →
4. **QLoRA 학습** → 5. **정량 평가(held-out test, base vs 파인튜닝)** → 6. Drive 저장

런타임: **수정 → 런타임 유형 변경 → T4 GPU** 로 먼저 설정하세요.

**논문급 설계 포인트**
- 누수 차단: base 쿼리 단위 train/val/**test** 분할. test는 학습에 안 쓴 **held-out 원본**.
- 평가: answer F1(토큰/문자), citation P/R/F1, format, abstain/hallucination, latency.
- 메모리 안전: 평가는 base/파인튜닝을 **각각 별도 프로세스**로 실행해 동시 로드 회피.

> repo에 **누수 차단 데이터셋(train/val/test + manifest)이 이미 포함**돼 있어,
> **2·3을 건너뛰고 1→4→5** 만 돌려도 됩니다. (재생성하려면 `UPSTAGE_API_KEY` 필요)

## 0. GPU 확인 (T4 인지 체크)

In [ ]:
!nvidia-smi

## 1. repo clone + 의존성 설치

In [ ]:
REPO_URL = "https://github.com/haneul-dev/recrag-generation-module.git"
BRANCH   = "main"

%cd /content
!rm -rf recrag-generation-module
!git clone -b $BRANCH --single-branch $REPO_URL
%cd recrag-generation-module
!pip -q install -r requirements.txt
!pip -q install -r finetune/requirements_finetune.txt

## 2. (선택) teacher API 키 — 학습셋을 재생성할 때만 필요
repo에 학습셋이 이미 있으므로 **재생성이 필요 없으면 2·3번을 건너뛰세요.**

In [ ]:
import os, getpass
os.environ["UPSTAGE_API_KEY"] = getpass.getpass("UPSTAGE_API_KEY (없으면 빈칸): ")

## 3. (선택) 누수 없는 데이터셋 재생성
repo에 누수 차단 데이터셋이 이미 있으므로 **재생성이 필요 없으면 2·3을 건너뛰세요.**
재생성하면 base 쿼리 단위 분할로 `sft_train/sft_val/test.jsonl` + `dataset_manifest.json` 을 새로 만듭니다.

In [ ]:
# (선택) 데이터셋 재생성 — UPSTAGE_API_KEY 필요. 누수차단 train/val/test 생성.
!python finetune/build_sft_dataset.py --config finetune/finetune_config.yaml
!echo "---"; wc -l finetune/data/sft_train.jsonl finetune/data/sft_val.jsonl finetune/data/test.jsonl

## 4. QLoRA 학습 (T4에서 수십 분 예상)
결과 어댑터: `finetune/outputs/qwen2.5-7b-recrag-qlora/`

In [ ]:
!python finetune/train_qlora.py --config finetune/finetune_config.yaml

## 5. 정량 평가 (held-out test, base vs 파인튜닝)
학습에 **쓰지 않은** test셋에서 두 모델을 평가하고 비교표를 출력합니다.
메모리 안전을 위해 base / 파인튜닝을 **각각 별도 프로세스**로 실행합니다(프로세스 종료 시 GPU 해제).

In [ ]:
# 1) base 평가  2) 파인튜닝 평가  3) 비교표 (각각 별도 프로세스 = 메모리 안전)
!python finetune/evaluate.py --config finetune/finetune_config.yaml --tag base
!python finetune/evaluate.py --config finetune/finetune_config.yaml --tag finetuned --adapter finetune/outputs/qwen2.5-7b-recrag-qlora
!python finetune/evaluate.py --compare finetune/outputs/eval/base.summary.json finetune/outputs/eval/finetuned.summary.json

## 6. Google Drive에 저장 (어댑터 + 평가결과 + 데이터셋)
Colab 런타임이 끊겨도 보존합니다. **꼭 실행하세요.**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DST = "/content/drive/MyDrive/RECRAG"
!mkdir -p "$DST/qlora_adapter" "$DST/eval" "$DST/data"
!cp -r finetune/outputs/qwen2.5-7b-recrag-qlora/* "$DST/qlora_adapter/"
!cp -r finetune/outputs/eval/* "$DST/eval/" 2>/dev/null
!cp finetune/data/sft_train.jsonl finetune/data/sft_val.jsonl finetune/data/test.jsonl finetune/data/dataset_manifest.json "$DST/data/"
print("저장 완료:", DST)